# Task 1 — Central Pipeline Notebook

This notebook runs the **complete Task 1 pipeline** end-to-end:

| Step | What happens |
|------|-------------|
| 0 | Load configuration from `config/config.json` |
| 1 | *(Optional)* Convert a PDF → Markdown using Ollama (`pdf_to_md`) |
| 2 | Run the heading-extraction pipeline on the Markdown file |

Run cells top-to-bottom.  
Toggle `RUN_PDF_CONVERSION` in **Cell 2** if you already have a Markdown file and want to skip Step 1.

---
## 0 · Setup — paths & configuration

In [1]:
import os, sys, json

# Make Task1 root importable from anywhere the notebook is launched
TASK1_ROOT = os.path.dirname(os.path.abspath("__file__"))
if TASK1_ROOT not in sys.path:
    sys.path.insert(0, TASK1_ROOT)

# ── Load config.json ──────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join(TASK1_ROOT, "config", "config.json")
with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    cfg = json.load(_f)

# Resolve paths relative to Task1 root
PDF_SOURCE   = cfg.get("pdf_source", "")
MARKDOWN_PATH = os.path.normpath(os.path.join(TASK1_ROOT, cfg["output"]["markdown_path"]))
OUTPUT_FILE   = os.path.normpath(os.path.join(TASK1_ROOT, cfg["output"]["output_file"]))
OLLAMA_MODEL  = cfg["model"]["model_id"]

if PDF_SOURCE and not os.path.isabs(PDF_SOURCE):
    PDF_SOURCE = os.path.normpath(os.path.join(TASK1_ROOT, PDF_SOURCE))

print("Config loaded successfully")
print(f"  PDF source    : {PDF_SOURCE}")
print(f"  Markdown path : {MARKDOWN_PATH}")
print(f"  Output file   : {OUTPUT_FILE}")
print(f"  Model         : {OLLAMA_MODEL}")

Config loaded successfully
  PDF source    : c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\book.pdf
  Markdown path : c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\Markdowns\sales_analysis_report.md
  Output file   : c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\output\md_json_outputs\sales_analysis_report4.json
  Model         : gemma2:27b


In [25]:
! pip install truststore

  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
Using cached truststore-0.10.4-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import truststore
truststore.inject_into_ssl()

---
## 1 · (Optional) PDF → Markdown conversion

Set `RUN_PDF_CONVERSION = True` to convert the PDF defined in `config.json → pdf_source`.  
Skip this step if you already have a Markdown file at `config.json → output.markdown_path`.

In [2]:
RUN_PDF_CONVERSION = True   # ← set to True to convert the PDF first

if RUN_PDF_CONVERSION:
    from utils.pdf_to_md import convert_pdf_to_markdown

    if not PDF_SOURCE or not os.path.isfile(PDF_SOURCE):
        raise FileNotFoundError(
            f"PDF not found at '{PDF_SOURCE}'. "
            "Check pdf_source in config/config.json."
        )

    convert_pdf_to_markdown(
        pdf_path=PDF_SOURCE,
        output_md_path=MARKDOWN_PATH,
        ollama_model=OLLAMA_MODEL,
        ollama_base_url=cfg.get("ollama_base_url", "http://localhost:11434"),
    )
else:
    print("PDF conversion skipped.")
    if os.path.isfile(MARKDOWN_PATH):
        print(f"  ✔ Markdown file found at: {MARKDOWN_PATH}")
    else:
        print(f"  ⚠ WARNING: Markdown file does NOT exist at: {MARKDOWN_PATH}")
        print("  → Set RUN_PDF_CONVERSION = True above to generate it from the PDF.")

c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\my_venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting conversion for: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\book.pdf
Using device: cuda


Recognizing tables: 100%|██████████| 1/1 [00:02<00:00,  2.47s/it]
Detecting bboxes: 0it [00:00, ?it/s]


Successfully converted PDF to markdown and saved as c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\Markdowns\book.md


---
## 2 · Heading extraction pipeline

In [33]:
! pip install langchain-text-splitters

  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl (187 kB)
Using cached jsonpointer-3.0.0-py2.py3-none-any.whl (7.6 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)

   ----------------------------------------  0/10 [zstandard]
   ---- -----------------------------------  1/10 [xxhash]
   -------- -------------------------------  2/10 [uuid-utils]
   ------------ ---------------------------  3/10 [orjson]
   ---------------- -----------------------  4/10 [jsonpointer]
 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
! pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   ---------------------------------------- 0/2 [et-xmlfile]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ----------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import importlib, sys, os
# Force-reload main and config so code/config changes take effect without a kernel restart
for _mod in list(sys.modules.keys()):
    if _mod in ('main', 'config') or _mod.startswith('config.'):
        del sys.modules[_mod]

import main as _main_module
importlib.reload(_main_module)
from main import process_markdown

if not os.path.isfile(MARKDOWN_PATH):
    raise FileNotFoundError(
        f"Markdown file not found at '{MARKDOWN_PATH}'.\n"
        "Run Step 1 (set RUN_PDF_CONVERSION = True) or place a .md file there."
    )

metrics = process_markdown(MARKDOWN_PATH)
print("\nPipeline complete!")

c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\my_venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  MARKDOWN HEADING EXTRACTION PIPELINE
  Model: gemma2:27b
  Input: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\Markdowns\sales_analysis_report.md

[Step 1] Reading Markdown text
--------------------------------------------------
  Document length: 5,448 characters
  Doc retrieval time: 0.00s

[Step 2] Splitting text into chunks
--------------------------------------------------
  Chunks created: 3

[Step 3] Extracting headings with LangExtract
--------------------------------------------------


LangExtract: Processing, current=2,152 chars, processed=0 chars:  [11:44]


  Chunk  1/3: 15 headings [LLM] in 710.83s


LangExtract: Processing, current=1,847 chars, processed=0 chars:  [14:40]


  Chunk  2/3: 18 headings [LLM] in 880.73s


LangExtract: Processing, current=483 chars, processed=0 chars:  [04:38]

  Chunk  3/3:  5 headings [LLM] in 278.69s
  Total LLM time: 1870.25s across 3 calls

[Step 4] Assembling per-chunk LangExtract outputs
--------------------------------------------------
  Chunks returned: 3
  Saved to: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\output\md_json_outputs\sales_analysis_report4.json

                    EXTRACTION METRICS
Document Statistics                               Value
-------------------------------------------------------
  Document Length                            5,448 chars
  Total Chunks Processed                              3
  Chunk Size                                       2500

Heading Extraction                                Count
-------------------------------------------------------
  Raw Headings Extracted                             38
  Valid Unique Headings                              38
  Rejected (False Pattern)                            0
  Rejected (Body Text)                           

---
## 3 · Inspect results

In [6]:
import json, os
# Re-read OUTPUT_FILE from config so it matches what the pipeline just saved
_cfg_path = os.path.join(TASK1_ROOT, "config", "config.json")
with open(_cfg_path, "r", encoding="utf-8") as _f:
    _cfg = json.load(_f)
OUTPUT_FILE = os.path.normpath(os.path.join(TASK1_ROOT, _cfg["output"]["output_file"]))
print(f"Reading: {OUTPUT_FILE}")

with open(OUTPUT_FILE, "r", encoding="utf-8") as _f:
    results = json.load(_f)

print(f"Total chunks in output: {len(results)}")
print("\n--- First chunk preview ---")
first = results[0] if results else {}
print(json.dumps({k: v for k, v in first.items() if k != "Text"}, indent=2, ensure_ascii=False))

Reading: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\output\Markdowns\notes2.json
Total chunks in output: 49

--- First chunk preview ---
{
  "chunk_id1": {
    "Main Heading 1": "Complete .....",
    "Sub Sub Sub Heading 1": "Chapter - 01 .....",
    "Sub Sub Sub Heading 2": "Introduction to Python .....",
    "Sub Sub Sub Heading 3": "What is Programming Language? .....",
    "Sub Sub Sub Heading 4": "What is Python? .....",
    "Sub Sub Sub Heading 5": "Popular programming languages .....",
    "Sub Sub Sub Heading 6": "Why Python? .....",
    "Sub Sub Sub Heading 7": "Python is Dynamically Typed Example .....",
    "Sub Sub Sub Heading 8": "Python - Easy to Read & Write ....."
  },
  "Metadata": "\"Sub Sub Sub heading\": \"Chapter - 01\", \"Sub Sub Sub heading\": \"Introduction to Python: ........- What is programming - What is Python - Popular programming languages - Why Python - Career with Python.........\", \"Sub Sub Sub heading\": \"What is Pr

In [2]:
! pip freeze > requirements313.txt